# init-process-group-nccl — ex1: init + destroy a process group with gloo

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `init-process-group-nccl`. Running the final beacon cell reports progress against the `Distributed: init_process_group nccl` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: init_process_group nccl` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`init-process-group-nccl`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "init-process-group-nccl"
DD_SUBTOPIC = "Distributed: init_process_group nccl"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## torch.distributed quick refresher

PyTorch's collective-communication library (`torch.distributed`, aliased `dist`) lets multiple processes coordinate over tensors. The standard workflow:

1. **Each rank** runs the same function, parameterized by `rank` and `world_size`. Rank 0 is conventionally the 'driver'.
2. **`dist.init_process_group(backend=...)`** establishes the rendezvous. Backends:
   - `'nccl'` — NVIDIA's GPU-to-GPU primitive. Used in ARENA's multi-GPU setup. Requires CUDA + one process per GPU.
   - `'gloo'` — CPU-friendly. What you'll use in these drills (Colab CPU runtimes have no real GPUs).
3. **Pin a device** per rank: `torch.device(f'cuda:{rank}')` so each process owns exactly one GPU.
4. **Collective ops** (`all_reduce`, `broadcast`, `send`, `recv`) operate in-place on tensors of identical shape across all ranks.
5. **`dist.destroy_process_group()`** tears down at the end.

**Two ways to launch multiple ranks:**
- `torch.multiprocessing.spawn(fn, args=(...), nprocs=world_size)` — what ARENA uses. Spawn requires the worker fn be importable (not defined in `__main__`/a notebook cell).
- `mp.get_context('fork').Process(target=fn, args=...)` — Linux-only but works with cell-defined fns. The drills use this in tests so the worker can stay in the cell.

**Two-rank trick.** Colab gives ~2 CPU cores, so `world_size=2` is the right scale: enough to exercise the protocol, cheap enough to finish in seconds.

### This drill's atom: `init_process_group`
ARENA's solution calls `dist.init_process_group(backend='nccl', rank=rank, world_size=world_size)` and pairs it with `dist.destroy_process_group()`. The drill below uses **`'gloo'`** (CPU) so you can run it on Colab, but the call pattern is the same.

### Exercise 1 — init + destroy a process group with gloo

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `dist.init_process_group` + `dist.destroy_process_group` to open and cleanly tear down a 2-rank `gloo` process group from a worker function.
> Keywords: init_process_group, gloo, nccl, destroy_process_group, lifecycle
> ```

**KCs targeted:** `init-process-group-call`, `destroy-process-group-pair`

Implement `ex1_worker(rank, world_size, port)`. The minimum-viable distributed worker:

1. Set the rendezvous env vars: `os.environ['MASTER_ADDR'] = '127.0.0.1'`, `os.environ['MASTER_PORT'] = str(port)`.
2. Call `dist.init_process_group(backend='gloo', rank=rank, world_size=world_size, timeout=datetime.timedelta(seconds=20))`.
3. After init, call `dist.get_rank()` and `dist.get_world_size()` and **print** them prefixed with `f'[rank {rank}] '` so the test harness can capture them.
4. Call `dist.destroy_process_group()` before returning.

The function takes a 3rd `port` arg so multiple tests can pick different ports and avoid collisions. The test spawns 2 forked procs and asserts both exit cleanly (exitcode 0).

**Why `gloo`, not `nccl`?** `nccl` requires real GPUs. Colab CPU runtimes have none. ARENA uses `'nccl'` because they're on multi-GPU boxes. The argument is literally the only thing that changes between the two backends.

In [ ]:
import os
import datetime
import torch.distributed as dist

def ex1_worker(rank: int, world_size: int, port: int) -> None:
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = str(port)
    dist.init_process_group(
        backend='gloo',
        rank=rank,
        world_size=world_size,
        timeout=datetime.timedelta(seconds=20),
    )
    print(f'[rank {rank}] dist.get_rank()={dist.get_rank()} dist.get_world_size()={dist.get_world_size()}')
    dist.destroy_process_group()


<details><summary>Solution</summary>

```python
import os
import datetime
import torch.distributed as dist

def ex1_worker(rank: int, world_size: int, port: int) -> None:
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = str(port)
    dist.init_process_group(
        backend='gloo',
        rank=rank,
        world_size=world_size,
        timeout=datetime.timedelta(seconds=20),
    )
    print(f'[rank {rank}] dist.get_rank()={dist.get_rank()} dist.get_world_size()={dist.get_world_size()}')
    dist.destroy_process_group()
```

**The four env vars `init_process_group` needs.** `MASTER_ADDR` + `MASTER_PORT` (rendezvous endpoint) are mandatory. `RANK` + `WORLD_SIZE` are also looked up from env if not passed as kwargs — the explicit-kwargs form (used here) is clearer.

**Why `timeout=20s` is generous.** The default is 30 minutes (yes, really). For a CPU test we want fast failure if rendezvous fails. Production code on slow networks uses the default.

**`destroy_process_group()` matters.** Without it, the port stays bound and the next init on the same port hangs. The third test case above (re-using port 29510) is the regression check.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()